In [ ]:
import os, re, unicodedata, json, math, random
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import VarianceThreshold

import joblib
import spacy
!pip install textstat
import textstat


import nltk
nltk.download('punkt', quiet=True)


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

nlp = spacy.load('en_core_web_sm')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.1 MB/s eta 0:00:00
Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def normalize_unicode_and_space(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace('\u2013', '-').replace('\u2014', ' - ')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
punct_space_re = re.compile(r"\s*([,.;:!?()\[\]{}\-—\"'“”‘’])\s*")

In [ ]:
def fix_punctuation_spacing(text: str) -> str:
    if not text:
        return ""
    text = punct_space_re.sub(r" \1 ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def basic_clean(text: str) -> str:
    t = normalize_unicode_and_space(text)
    t = fix_punctuation_spacing(t)
    return t

def sentence_split(text: str):
    text = normalize_unicode_and_space(text)
    try:
        doc = nlp(text)
        sents = [s.text.strip() for s in doc.sents]
        if sents:
            return sents
    except Exception:
        pass
    # fallback nltk or regex
    try:
        from nltk import sent_tokenize
        sents = sent_tokenize(text)
        if sents:
            return sents
    except Exception:
        pass
    sents = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sents if s.strip()]

def tokenize_with_spacy(text: str):
    text = basic_clean(text)
    doc = nlp(text)
    tokens = [tok for tok in doc if not tok.is_space]
    words = [tok.text for tok in tokens if tok.is_alpha]
    return tokens, words

def linguistic_annotations(text: str):
    text = basic_clean(text)
    doc = nlp(text)
    pos_dep = [(tok.text, tok.lemma_, tok.pos_, tok.tag_, tok.dep_) for tok in doc]
    ents = [(ent.text, ent.label_) for ent in doc.ents] if hasattr(doc, 'ents') else []
    sents = [s.text for s in doc.sents]
    return {'pos_dep': pos_dep, 'entities': ents, 'sents': sents}

def count_dialogue_markers(text: str) -> int:
    if not text: return 0
    return text.count('"') + text.count("'") + text.count('—') + text.count('-')

In [ ]:
class StylometricFeatureExtractor:
    def __init__(self, enable_readability=True):
        self.enable_readability = enable_readability

    def _approx_syllable_count(self, w):
        return max(1, len(re.findall(r'[aeiouyAEIOUY]+', w)))

    def extract_all_features(self, text):
        text_orig = "" if text is None else str(text)
        text = basic_clean(text_orig)

        features = {}
        sents = sentence_split(text)
        tokens, words = tokenize_with_spacy(text)
        num_words = len(words)
        features['char_count'] = len(text)
        features['word_count'] = num_words
        features['sentence_count'] = max(1, len(sents))
        features['avg_word_length'] = float(np.mean([len(w) for w in words])) if words else 0.0
        features['avg_sentence_length'] = float(num_words) / features['sentence_count'] if features['sentence_count']>0 else 0.0

        sent_word_counts = []
        try:
            doc = nlp(text)
            for s in doc.sents:
                sent_word_counts.append(len([t for t in s if t.is_alpha]))
        except Exception:
            sent_word_counts = [len(s.split()) for s in sents]
        if sent_word_counts:
            features['sent_len_mean'] = float(np.mean(sent_word_counts))
            features['sent_len_std'] = float(np.std(sent_word_counts))
            features['sent_len_max'] = int(np.max(sent_word_counts))
        else:
            features['sent_len_mean'] = features['sent_len_std'] = 0.0
            features['sent_len_max'] = 0

        words_lower = [w.lower() for w in words]
        if words_lower:
            freq = Counter(words_lower)
            features['type_token_ratio'] = len(set(words_lower)) / len(words_lower)
            features['hapax_legomena_ratio'] = sum(1 for v in freq.values() if v == 1) / len(words_lower)
            try:
                M1 = len(words_lower)
                features['yules_k'] = (1_000_000 * (sum(v*v for v in freq.values()) - M1)) / (M1*M1)
            except Exception:
                features['yules_k'] = 0.0
        else:
            features['type_token_ratio'] = 0.0
            features['hapax_legomena_ratio'] = 0.0
            features['yules_k'] = 0.0


        denom = max(1, features['char_count'])
        features['comma_freq'] = text.count(',') / denom * 1000
        features['period_freq'] = text.count('.') / denom * 1000
        features['exclamation_freq'] = text.count('!') / denom * 1000
        features['question_freq'] = text.count('?') / denom * 1000
        features['quote_freq'] = (text.count('"') + text.count("'")) / denom * 1000


        features['dialogue_markers'] = count_dialogue_markers(text) / max(1, num_words)
        features['first_person'] = sum(1 for w in words_lower if w in ['i','me','my','we','us','our']) / max(1, len(words_lower))


        try:
            ann = linguistic_annotations(text)
            pos_dep = ann['pos_dep']
            total_pos = max(1, len(pos_dep))
            pos_counts = Counter([p[2] for p in pos_dep])  # coarse POS
            for p in ['NOUN','VERB','ADJ','ADV','PRON','ADP','CONJ','DET','NUM','PUNCT']:
                features[f'pos_{p}'] = pos_counts.get(p, 0) / total_pos
            deps = [p[4] for p in pos_dep]
            features['passive_ratio'] = (deps.count('auxpass') + sum(1 for i,p in enumerate(pos_dep[:-1]) if p[2]=='AUX' and pos_dep[i+1][3]=='VBN')) / total_pos

            try:
                doc_full = nlp(text)
                depths = []
                for tok in doc_full:
                    depth = 0
                    node = tok
                    while node.head is not node:
                        depth += 1
                        node = node.head
                        if depth > 200:
                            break
                    depths.append(depth)
                features['parse_depth_mean'] = float(np.mean(depths)) if depths else 0.0
                features['parse_depth_max'] = int(np.max(depths)) if depths else 0
            except Exception:
                features['parse_depth_mean'] = 0.0
                features['parse_depth_max'] = 0
        except Exception:
            for p in ['NOUN','VERB','ADJ','ADV','PRON','ADP','CONJ','DET','NUM','PUNCT']:
                features[f'pos_{p}'] = 0.0
            features['passive_ratio'] = 0.0
            features['parse_depth_mean'] = 0.0
            features['parse_depth_max'] = 0


        if self.enable_readability:
            try:
                features['flesch_reading_ease'] = textstat.flesch_reading_ease(text)
                features['flesch_kincaid_grade'] = textstat.flesch_kincaid_grade(text)
            except Exception:
                syllables = sum(self._approx_syllable_count(w) for w in words_lower) if words_lower else 0
                features['flesch_reading_ease'] = 206.835 - 1.015 * (num_words / max(1, features['sentence_count'])) - 84.6 * (syllables / max(1, num_words))
                features['flesch_kincaid_grade'] = 0.0

        return features

In [ ]:
DATA_PATH = Path('/content/drive/MyDrive/NLP/final_dataset.csv')

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
df.sample(20)

,text,label_name
361,I’ve never been excited for a DC film until now.,Normal
158,"OFT, in the silence of the night,\n When the...",poetic
480,"This looks quite fantastic, the scale the emot...",Normal
641,Softness is underestimated. People assume gent...,philosophical
275,"“No—not really,” said Mrs. Mannering. “Some of...",narrative
362,The effects look very good. They make this mov...,Normal
310,Notable Australian examples include Queensland...,journalistic
199,"Oh, I can smile for you, and tilt my head, \nA...",poetic
523,I've seen Dwayne Johnson portray as the hero l...,Normal
90,We derive an Abelian-like Ward identity in c...,Academic


In [ ]:
df = df.dropna(subset=['text','label_name']).reset_index(drop=True)
df['word_count'] = df['text'].astype(str).apply(lambda x: len(str(x).split()))
df = df[df['word_count'] >= 5].reset_index(drop=True)
print("After filtering short docs:", df.shape)

After filtering short docs: (654, 3)


In [ ]:
df.shape

(654, 3)

In [ ]:
df.head()

,text,label_name,word_count
0,Let C be a soluble smooth genus one curve ov...,Academic,121
1,"Due to the non-stationary nature, the distribu...",Academic,232
2,Blind and low vision (BLV) internet users ac...,Academic,187
3,I introduce a generic method for inference o...,Academic,115
4,"Nowadays, fast delivery services have create...",Academic,130


In [ ]:
label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['label_name'])
num_classes = len(label_encoder.classes_)
print("Classes:", list(label_encoder.classes_), "| Num classes:", num_classes)

Classes: ['Academic', 'Normal', 'journalistic', 'narrative', 'philosophical', 'poetic'] | Num classes: 6


In [ ]:
extractor = StylometricFeatureExtractor(enable_readability=True)
FEATURES_CACHE = Path('/content/drive/MyDrive/NLP/stylometry_features.parquet')

In [ ]:
if FEATURES_CACHE.exists():
    feats_df = pd.read_parquet(FEATURES_CACHE)
    print("Loaded cached features:", feats_df.shape)
else:
    rows = []
    for i, txt in tqdm(enumerate(df['text'].astype(str)), total=len(df), desc='Extract features'):
        try:
            feats = extractor.extract_all_features(txt)
        except Exception as e:
            print("Feature extraction error at idx", i, e)
            feats = {}
        feats['idx'] = i
        rows.append(feats)
    feats_df = pd.DataFrame(rows).set_index('idx').fillna(0)
    feats_df.to_parquet(FEATURES_CACHE)
    print("Saved features to cache:", FEATURES_CACHE)


Extract features:   0%|          | 0/654 [00:00<?, ?it/s]

Saved features to cache: /content/drive/MyDrive/NLP/stylometry_features.parquet


In [ ]:
big = pd.concat([df.reset_index(drop=True), feats_df.reset_index(drop=True)], axis=1).fillna(0)
exclude_cols = {'text','label','label_id','word_count'}
stylo_cols = [c for c in big.columns if c not in exclude_cols and big[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
print("Number of stylometric columns:", len(stylo_cols))

Number of stylometric columns: 32


In [ ]:
big.head()

,text,label_name,word_count,label_id,char_count,word_count,sentence_count,avg_word_length,avg_sentence_length,sent_len_mean,...,pos_ADP,pos_CONJ,pos_DET,pos_NUM,pos_PUNCT,passive_ratio,parse_depth_mean,parse_depth_max,flesch_reading_ease,flesch_kincaid_grade
0,Let C be a soluble smooth genus one curve ov...,Academic,121,0,670,114,6,4.552632,19.000000,19.000000,...,0.142857,0.0,0.090226,0.060150,0.090226,0.000000,201.0,201,47.785198,11.480000
1,"Due to the non-stationary nature, the distribu...",Academic,232,0,1839,244,10,6.270492,24.400000,24.400000,...,0.104693,0.0,0.086643,0.003610,0.119134,0.007220,201.0,201,2.814082,18.928459
2,Blind and low vision (BLV) internet users ac...,Academic,187,0,1472,201,9,5.920398,22.333333,22.333333,...,0.082988,0.0,0.045643,0.004149,0.153527,0.000000,201.0,201,17.658352,16.399770
3,I introduce a generic method for inference o...,Academic,115,0,818,118,6,5.720339,19.666667,19.666667,...,0.106870,0.0,0.068702,0.007634,0.099237,0.030534,201.0,201,20.541130,15.280000
4,"Nowadays, fast delivery services have create...",Academic,130,0,1018,138,6,5.898551,23.000000,23.000000,...,0.042169,0.0,0.108434,0.006024,0.156627,0.042169,201.0,201,9.385652,17.664058


In [ ]:
df.shape

(654, 4)

In [ ]:
df.head()

,text,label_name,word_count,label_id
0,Let C be a soluble smooth genus one curve ov...,Academic,121,0
1,"Due to the non-stationary nature, the distribu...",Academic,232,0
2,Blind and low vision (BLV) internet users ac...,Academic,187,0
3,I introduce a generic method for inference o...,Academic,115,0
4,"Nowadays, fast delivery services have create...",Academic,130,0


In [ ]:
scaler = StandardScaler()
stylo_matrix = scaler.fit_transform(big[stylo_cols].values)
# optional variance selector
vt = VarianceThreshold(threshold=0.0)
stylo_matrix = vt.fit_transform(stylo_matrix)
print("Stylometric matrix shape:", stylo_matrix.shape)

Stylometric matrix shape: (654, 29)


In [ ]:
os.makedirs('/content/drive/MyDrive/NLP/artifacts', exist_ok=True)
joblib.dump(scaler, '/content/drive/MyDrive/NLP/artifacts/stylo_scaler.joblib')
joblib.dump(vt, '/content/drive/MyDrive/NLP/artifacts/stylo_var_threshold.joblib')
joblib.dump(label_encoder, '/content/drive/MyDrive/NLP/artifacts/label_encoder.joblib')

['/content/drive/MyDrive/NLP/artifacts/label_encoder.joblib']

In [ ]:
EMB_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'  # multilingual; change if you want
print("Loading embedder:", EMB_MODEL)
embedder = SentenceTransformer(EMB_MODEL)
embedder = embedder.to(DEVICE)

EMB_CACHE = Path('/content/drive/MyDrive/NLP/document_embeddings.npy')
if EMB_CACHE.exists():
    doc_embs = np.load(EMB_CACHE)
    print("Loaded cached embeddings:", doc_embs.shape)
else:
    texts = big['text'].astype(str).tolist()
    doc_embs = embedder.encode(texts, show_progress_bar=True, batch_size=32)
    np.save(EMB_CACHE, doc_embs)
    print("Saved embeddings to cache:", EMB_CACHE)

Loading embedder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Saved embeddings to cache: /content/drive/MyDrive/NLP/document_embeddings.npy


In [ ]:
class StylometryDataset(Dataset):
    def __init__(self, stylo, emb, labels, indices):
        self.stylo = torch.tensor(stylo[indices]).float()
        self.emb = torch.tensor(emb[indices]).float()
        self.labels = torch.tensor(labels[indices]).long()
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {'stylo': self.stylo[idx], 'emb': self.emb[idx], 'label': self.labels[idx]}

indices = np.arange(len(big))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED, stratify=big['label_id'])
train_idx, val_idx = train_test_split(train_idx, test_size=0.125, random_state=SEED, stratify=big.loc[train_idx,'label_id'])

train_ds = StylometryDataset(stylo_matrix, doc_embs, big['label_id'].values, train_idx)
val_ds = StylometryDataset(stylo_matrix, doc_embs, big['label_id'].values, val_idx)
test_ds = StylometryDataset(stylo_matrix, doc_embs, big['label_id'].values, test_idx)

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)

In [ ]:
class FusionClassifier(nn.Module):
    def __init__(self, stylo_dim, emb_dim, hidden=512, num_classes=2, dropout=0.3):
        super().__init__()
        self.bn = nn.BatchNorm1d(stylo_dim + emb_dim)
        self.fc1 = nn.Linear(stylo_dim + emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden//2)
        self.out = nn.Linear(hidden//2, num_classes)
        self.drop = nn.Dropout(dropout)
    def forward(self, stylo, emb):
        x = torch.cat([stylo, emb], dim=1)
        x = self.bn(x)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        x = self.drop(x)
        return self.out(x)

In [ ]:
stylo_dim = stylo_matrix.shape[1]
emb_dim = doc_embs.shape[1]
model = FusionClassifier(stylo_dim, emb_dim, hidden=512, num_classes=num_classes, dropout=0.3).to(DEVICE)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

Model parameters: 345664


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    losses = []
    all_preds, all_labels = [], []
    for batch in loader:
        sty = batch['stylo'].to(device)
        emb = batch['emb'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(sty, emb)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
        preds = logits.argmax(1).detach().cpu().numpy()
        all_preds.extend(preds.tolist()); all_labels.extend(labels.detach().cpu().numpy().tolist())
    return np.mean(losses), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro')

def evaluate(model, loader, device):
    model.eval()
    losses = []; all_preds=[]; all_labels=[]
    with torch.no_grad():
        for batch in loader:
            sty = batch['stylo'].to(device)
            emb = batch['emb'].to(device)
            labels = batch['label'].to(device)
            logits = model(sty, emb)
            loss = criterion(logits, labels)
            losses.append(loss.item())
            preds = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds.tolist()); all_labels.extend(labels.cpu().numpy().tolist())
    return np.mean(losses), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels

In [ ]:
EPOCHS = 20
best_val_f1 = -1.0
CKPT = '/content/drive/MyDrive/NLP/artifacts/best_fusion.pt'

In [ ]:
history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[]}
for epoch in range(EPOCHS):
    tr_loss, tr_acc, tr_f1 = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, DEVICE)
    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_f1'].append(tr_f1); history['val_f1'].append(val_f1)
    print(f"Epoch {epoch+1}/{EPOCHS} | tr_loss {tr_loss:.4f} tr_f1 {tr_f1:.4f} | val_loss {val_loss:.4f} val_f1 {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'model_state': model.state_dict(), 'stylo_cols': stylo_cols, 'label_encoder': label_encoder}, CKPT)
        print("Saved best checkpoint.")

print("Training complete. Best val F1:", best_val_f1)

# ------------------ FINAL EVALUATION ------------------
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, DEVICE)
test_loss, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, DEVICE)
print("Test Acc: {:.2f}% | Test F1: {:.4f}".format(test_acc*100, test_f1))
print("\nClassification report (test):\n", classification_report(test_labels, test_preds, target_names=label_encoder.classes_))

# save artifacts
np.save('/content/drive/MyDrive/NLP/artifacts/document_embeddings.npy', doc_embs)
joblib.dump(stylo_cols, '/content/drive/MyDrive/NLP/artifacts/stylo_columns.pkl')
torch.save(model.state_dict(), '/content/drive/MyDrive/NLP/artifacts/final_model_state.pt')
print("Saved artifacts to /content/drive/MyDrive/NLP/artifacts/")

Epoch 1/20 | tr_loss 0.0042 tr_f1 1.0000 | val_loss 0.1249 val_f1 0.9371
Epoch 2/20 | tr_loss 0.0091 tr_f1 0.9967 | val_loss 0.1403 val_f1 0.9371
Epoch 3/20 | tr_loss 0.0078 tr_f1 1.0000 | val_loss 0.1286 val_f1 0.9371
Epoch 4/20 | tr_loss 0.0032 tr_f1 1.0000 | val_loss 0.1419 val_f1 0.9371
Epoch 5/20 | tr_loss 0.0034 tr_f1 1.0000 | val_loss 0.1312 val_f1 0.9371
Epoch 6/20 | tr_loss 0.0018 tr_f1 1.0000 | val_loss 0.1384 val_f1 0.9371
Epoch 7/20 | tr_loss 0.0199 tr_f1 0.9967 | val_loss 0.1685 val_f1 0.9371
Epoch 8/20 | tr_loss 0.0036 tr_f1 1.0000 | val_loss 0.1512 val_f1 0.9371
Epoch 9/20 | tr_loss 0.0022 tr_f1 1.0000 | val_loss 0.1434 val_f1 0.9371
Epoch 10/20 | tr_loss 0.0014 tr_f1 1.0000 | val_loss 0.1476 val_f1 0.9371
Epoch 11/20 | tr_loss 0.0038 tr_f1 1.0000 | val_loss 0.1500 val_f1 0.9371
Epoch 12/20 | tr_loss 0.0019 tr_f1 1.0000 | val_loss 0.1616 val_f1 0.9371
Epoch 13/20 | tr_loss 0.0017 tr_f1 1.0000 | val_loss 0.1469 val_f1 0.9371
Epoch 14/20 | tr_loss 0.0014 tr_f1 1.0000 | val

In [ ]:
def predict_style(text, model, scaler, vt, embedder, stylo_cols, label_encoder):
    model.eval()
    feats = extractor.extract_all_features(text)
    stylo_vec = np.array([feats.get(c, 0.0) for c in stylo_cols]).reshape(1,-1)
    stylo_vec = scaler.transform(stylo_vec)
    stylo_vec = vt.transform(stylo_vec)
    emb = embedder.encode([text])
    with torch.no_grad():
        sty_t = torch.tensor(stylo_vec).float().to(DEVICE)
        emb_t = torch.tensor(emb).float().to(DEVICE)
        logits = model(sty_t, emb_t)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred = logits.argmax(1).cpu().numpy()[0]
    return label_encoder.inverse_transform([pred])[0], float(probs[pred]), probs

sample = "In the evening, she whispered to the moon about the sea of memories."
label, conf, probs = predict_style(sample, model, scaler, vt, embedder, stylo_cols, label_encoder)
print("Sample prediction:", label, conf)


Sample prediction: poetic 0.6104555726051331
